# Les k-Plus Proches Voisins : en pratique

Dans le TP4 nous avons implémenté quelques méthodes de base en Machine Learning (Apprentissage Automatique) :
- découpage train/test d'un dataset
- normalisation
- classification par k-PPV (méthode de base et variante pondérée)
- évaluation d'un classifieur

Le fait d'en écrire le code a permis d'en maîtriser le fonctionnement. Cependant, à l'avenir vous devrez utiliser des librairies qui proposent ces méthodes déjà implémentées.

> [Scikit-learn](https://scikit-learn.org/) est une bibliothèque libre Python destinée à l'apprentissage automatique.
>
> Nous y retrouverons notamment :
> - la méthode `train_test_split()` pour découper un dataset en deux sous-ensembles d'entraînement et de test
> - les méthodes `MinMaxScaler()` et `StandardScaler()` pour normaliser le dataset
> - la méthode `KNeighborsClassifier()` pour prédire par k-PPV
>


Vous commencerez par installer la bibliothèque sur votre machine : `pip install scikit-learn`

---

### Exercice 1 : prise en main de Scikit-learn

1. Comme dans le TP4, chargez les données `zoo.csv` dans un Dataframe puis en extraire deux tableaux NumPy `X` (matrice 100x16) et `y` (vecteur des 100 labels de classe)

In [2]:
from pandas import *

df = read_csv("donnees/zoo.csv")
df.head()



test = df.iloc[-1:]
result = test['type']
test.drop(["type", "Name"], axis=1, inplace=True)

df.drop(df.index[-1], inplace=True)

print(test)

X = df.drop(["type", "Name"], axis=1)
y = df["type"]


    hair  feathers  eggs  milk  airbone  aquatic  predator  toothed  backbone  \
99     0         1     1     0        1        0         0        0         1   

    breathes  venomous  fins  legs  tail  domestic  size  
99         1         0     0     2     1         0  14.0  


/tmp/ipykernel_58524/1893509660.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test.drop(["type", "Name"], axis=1, inplace=True)


2. Procédez au découpage du dataset (`X` et `y`) en deux sous-ensembles de même taille (`X_train`, `X_test`, `y_train`, `y_test`) en utilisant la méthode `train_test_split()` de Scikit-learn (consultez la documentation). Puis vérifiez l'équilibre des classes entre les deux sous-ensembles.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (66, 16)
X_test shape: (33, 16)
y_train shape: (66,)
y_test shape: (33,)


3. Normalisez les données en utilisant au choix l'une des méthodes suivantes : `MinMaxScaler()` ou `StandardScaler()` (consultez la doc)

In [4]:
# normalisation min/max z = (x-min)/(max-min)
from sklearn.preprocessing import MinMaxScaler

def minMaxScaler(X_train):
    scaler = MinMaxScaler()
    scaler.fit(X_train)
    return scaler.transform(X_train)


In [5]:
# normalisation centrée réduite z = (x-u)/s avec u la moyenne et s l'écart type
from sklearn.preprocessing import StandardScaler

def standardScaler(X_train):
    scaler = StandardScaler()
    scaler.fit(X_train)
    return scaler.transform(X_train)

4. Utilisez la classe `KNeighborsClassifier` de Scikit-learn pour prédire les classes de chaque données de l'ensemble `X_test` par la méthode k-PPV (non-pondérée) avec $k=2$ (consultez la documentation)

In [6]:
from sklearn.neighbors import KNeighborsClassifier

neigh = KNeighborsClassifier(n_neighbors=3)
neigh.fit(X, y)
print(neigh.predict(test)==result)

99    True
Name: type, dtype: bool


5. Evaluez ce classifieur en calculant le taux d'erreurs

In [7]:


def Knn_error(X_train, X_test, y_train, y_test, k, scaler='standard'):
    X_scaler = standardScaler(X_train) if scaler == 'standard' else minMaxScaler(X_train)
    neigh = KNeighborsClassifier(n_neighbors=k)
    neigh.fit(X_scaler, y_train)
    
    return neigh.score(X_test, y_test)


# print(Knn_error(X_train, X_test, y_train, y_test, 3))
# print(Knn_error(X_train, X_test, y_train, y_test, 3, scaler='minmax'))







def knn_error_from_dataset(filename, result_label , scaler='standard', k=3):
    df = read_csv(filename)
    X = df.drop(result_label, axis=1)
    y = df[result_label]
    X = df.select_dtypes(exclude=['object', 'string'])
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
    return Knn_error(X_train, X_test, y_train, y_test, k, scaler)



print(knn_error_from_dataset("donnees/zoo.csv", "type"))
print(knn_error_from_dataset("donnees/zoo.csv","type",  scaler='minmax'))

print(knn_error_from_dataset("donnees/Iris.csv" , "Species"))
print(knn_error_from_dataset("donnees/Iris.csv","Species", scaler='minmax'))

print(knn_error_from_dataset("donnees/card_transdata.csv" , "fraud"))
print(knn_error_from_dataset("donnees/card_transdata.csv", "fraud", scaler='minmax'))

    

0.48484848484848486
0.5151515151515151
0.32
0.32


/usr/lib/python3/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(
/usr/lib/python3/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


0.8618878787878788


/usr/lib/python3/dist-packages/sklearn/base.py:486: UserWarning: X has feature names, but KNeighborsClassifier was fitted without feature names
  warnings.warn(


0.9223272727272728


6. Modifiez les paramètres du classifieur (question 4) pour tester (et évaluer) différentes méthodes de prédication :
- différentes valeurs pour $k$
- avec ou sans normalisation des données
- variante pondérée de k-PPV
- etc.

---

### Exercice 2 : pour vous entraîner... et aller plus loin

Vous trouverez deux nouveaux datasets dans le repertoire `donnees` :
- le dataset `iris` : contenant les descriptions de fleurs (iris) selon 4 descripteurs numériques. La tâche de prédiction porte sur le type d'Iris (3 catégories).

- le dataset `card_transdata` : contenant les descriptions de transactions par cartes bancaires selont 7 descripteurs numériques ou booléens. La tâche de prédiction consiste à détecter les fraudes (2 catégories).

Votre travail consiste, pour chacune des deux tâches de prédiction ci-dessus, à proposer un classifieur de type k-PPV le plus efficace possible. 